# Введение в MapReduce модель на Python


In [50]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator
from itertools import groupby

In [51]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [52]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [53]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [54]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [55]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [56]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [57]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [58]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [59]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [60]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [61]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [62]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [63]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [64]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.2420564401289043)),
 (1, np.float64(1.2420564401289043)),
 (2, np.float64(1.2420564401289043)),
 (3, np.float64(1.2420564401289043)),
 (4, np.float64(1.2420564401289043))]

## Inverted index

In [65]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [66]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [67]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [68]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [])]

## TeraSort

In [69]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.02869588867576356)),
   (None, np.float64(0.05123740849329417)),
   (None, np.float64(0.10028109930145168)),
   (None, np.float64(0.10567564732635182)),
   (None, np.float64(0.12525781779462242)),
   (None, np.float64(0.1316099966667441)),
   (None, np.float64(0.2735744453089902)),
   (None, np.float64(0.30758420877760917)),
   (None, np.float64(0.352471786572615)),
   (None, np.float64(0.37213730053308836)),
   (None, np.float64(0.4038134159907345)),
   (None, np.float64(0.4091013274333739)),
   (None, np.float64(0.48073331984934786)),
   (None, np.float64(0.48530539084329183)),
   (None, np.float64(0.4882134850215296))]),
 (1,
  [(None, np.float64(0.5199832292711934)),
   (None, np.float64(0.5516772555387035)),
   (None, np.float64(0.6344583017881884)),
   (None, np.float64(0.680714062112336)),
   (None, np.float64(0.6810904363075801)),
   (None, np.float64(0.6982713463809552)),
   (None, np.float64(0.7232827977283218)),
   (None, np.float64(0.733032827092

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [70]:
def MAP(_, row):
    yield ("max", row)

def REDUCE(key, rows):
    return max(rows)

def RECORDREADER():
    return [(None, num) for num in input_collection]

def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]

def groupbykey(items):
    items.sort(key=lambda x: x[0])
    return [(key, [v for k, v in group]) for key, group in groupby(items, key=lambda x: x[0])]

def MapReduce(reader, map_func, reduce_func):
    records = reader()
    mapped = flatten([list(map_func(*record)) for record in records])
    grouped = groupbykey(mapped)
    return reduce_func(*grouped[0])  # Сразу возвращаем результат REDUCE

input_collection = [5, 2, 8, 1, 9, 3, 7, 4, 6]
output = MapReduce(RECORDREADER, MAP, REDUCE)
print(output)

9


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [71]:
def MAP(_, row):
    yield ("avg", (row, 1))

def REDUCE(key, rows):
    total_sum = 0
    total_count = 0
    for value, count in rows:
        total_sum += value
        total_count += count
    return total_sum / total_count

def RECORDREADER():
    return [(None, num) for num in input_collection]

def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]

def groupbykey(items):
    items.sort(key=lambda x: x[0])
    return [(key, [v for k, v in group]) for key, group in groupby(items, key=lambda x: x[0])]

def MapReduce(reader, map_func, reduce_func):
    records = reader()
    mapped = flatten([list(map_func(*record)) for record in records])
    grouped = groupbykey(mapped)
    return reduce_func(*grouped[0])

input_collection = [5, 2, 8, 1, 9, 3, 7, 4, 6]
output = MapReduce(RECORDREADER, MAP, REDUCE)
print(output)

5.0


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [72]:
data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(key, row):
     yield (key, row)

def RECORDREADER(key: str):
    for user in data:
        yield (getattr(user, key), user)

def flatten(iterable):
    for item in iterable:
        for subitem in item:
            yield subitem

def groupByKey(iterable):
    groups = {}
    for key, value in sorted(iterable, key=lambda x: x[0]):
        if key not in groups.keys():
            groups[key] = []
        groups[key].append(value)
    return groups.items()

print("Группировка по возрасту:")
result1 = list(groupByKey(flatten(map(lambda x: MAP(*x), RECORDREADER("age")))))
print(result1)
print()

Группировка по возрасту:
[(25, [User(id=1, age=25, social_contacts=240, gender='female'), User(id=2, age=25, social_contacts=500, gender='female')]), (33, [User(id=3, age=33, social_contacts=800, gender='female')]), (55, [User(id=0, age=55, social_contacts=20, gender='male')])]



### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [73]:
data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=3, age=33, gender='female', social_contacts=800)  # дубликат
]

def MAP(_, row):
    yield (row, None)

def REDUCE(key, rows):
    return key

def RECORDREADER():
    return [(None, user) for user in data]

def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]

def groupByKey(items):
    groups = {}
    for key, value in sorted(items, key=lambda x: x[0]):
        if key not in groups:
            groups[key] = []
        groups[key].append(value)
    return [(k, v) for k, v in groups.items()]

def MapReduce(reader, map_func, reduce_func):
    records = reader()
    mapped = flatten([list(map_func(*record)) for record in records])
    grouped = groupByKey(mapped)
    return [reduce_func(key, values) for key, values in grouped]

result = MapReduce(RECORDREADER, MAP, REDUCE)
print(result)

[User(id=0, age=55, social_contacts=20, gender='male'), User(id=1, age=25, social_contacts=240, gender='female'), User(id=3, age=33, social_contacts=800, gender='female')]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [74]:
data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row):
    C = (row.age > 30)
    if C:
        yield (row, row)

def REDUCE(key, rows):
    return key

def RECORDREADER():
    return [(None, user) for user in data]

def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]

def groupByKey(items):
    groups = {}
    for key, value in sorted(items, key=lambda x: x[0]):
        if key not in groups:
            groups[key] = []
        groups[key].append(value)
    return [(k, v) for k, v in groups.items()]

def MapReduce(reader, map_func, reduce_func):
    records = reader()
    mapped = flatten([list(map_func(*record)) for record in records])
    grouped = groupByKey(mapped)
    return [reduce_func(key, values) for key, values in grouped]

# возраст > 30
result = MapReduce(RECORDREADER, MAP, REDUCE)
print("Кортежи, удовлетворяющие предикату (возраст > 30):")
print(result)

Кортежи, удовлетворяющие предикату (возраст > 30):
[User(id=0, age=55, social_contacts=20, gender='male'), User(id=3, age=33, social_contacts=800, gender='female')]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [75]:
class UserProjection(NamedTuple):
  id: int
  age: int

data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=33, gender='female', social_contacts=800),
    User(id=3, age=35, gender='female', social_contacts=800)
]

def RECORDREADER():
  for user in data:
    yield (user.id, user)

def MAP(userId:int, user:User):
  newUser = UserProjection(userId, user.age)
  yield(newUser, newUser)

def REDUCE(key: UserProjection, usersList: list):
    yield (key, key)

def MapReduce(reader, map_func, reduce_func):
    records = reader()
    mapped = flatten([list(map_func(*record)) for record in records])
    grouped = groupByKey(mapped)
    return [next(reduce_func(key, values)) for key, values in grouped]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(UserProjection(id=0, age=55), UserProjection(id=0, age=55)),
 (UserProjection(id=1, age=25), UserProjection(id=1, age=25)),
 (UserProjection(id=2, age=33), UserProjection(id=2, age=33)),
 (UserProjection(id=3, age=35), UserProjection(id=3, age=35))]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [76]:
data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=33, gender='female', social_contacts=800),
    User(id=3, age=35, gender='female', social_contacts=800)
]

def MAP(_, row):
    yield (row, row)

def REDUCE(key, rows):
    return (key, key)

def RECORDREADER():
    return [(None, user) for user in data]

def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]

def groupByKey(items):
    groups = {}
    for key, value in sorted(items, key=lambda x: x[0]):
        if key not in groups:
            groups[key] = []
        groups[key].append(value)
    return [(k, v) for k, v in groups.items()]

def MapReduce(reader, map_func, reduce_func):
    records = reader()
    mapped = flatten([list(map_func(*record)) for record in records])
    grouped = groupByKey(mapped)
    return [reduce_func(key, values) for key, values in grouped]

output = MapReduce(RECORDREADER, MAP, REDUCE)
print(output)

[(User(id=0, age=55, social_contacts=20, gender='male'), User(id=0, age=55, social_contacts=20, gender='male')), (User(id=1, age=25, social_contacts=240, gender='female'), User(id=1, age=25, social_contacts=240, gender='female')), (User(id=2, age=33, social_contacts=800, gender='female'), User(id=2, age=33, social_contacts=800, gender='female')), (User(id=3, age=35, social_contacts=800, gender='female'), User(id=3, age=35, social_contacts=800, gender='female'))]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [77]:
data_1 = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=33, gender='female', social_contacts=800),
    User(id=3, age=35, gender='female', social_contacts=800)
]

data_2 = [
    User(id=4, age=55, gender='male', social_contacts=20),
    User(id=5, age=25, gender='female', social_contacts=240),
    User(id=6, age=33, gender='female', social_contacts=800),
    User(id=0, age=55, gender='male', social_contacts=20)
]

data = [data_1, data_2]

def RECORDREADER():
    for lists in data:
        for user in lists:
            yield (user.id, user)

def MAP(userId: int, user: User):
    yield (user, user)

def REDUCE(key: User, usersList: list):
    if len(usersList) == 2:
        return (key, key)  # убрали yield

def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]

def groupByKey(items):
    groups = {}
    for key, value in sorted(items, key=lambda x: x[0]):
        if key not in groups:
            groups[key] = []
        groups[key].append(value)
    return [(k, v) for k, v in groups.items()]

def MapReduce(reader, map_func, reduce_func):
    records = reader()
    mapped = flatten([list(map_func(*record)) for record in records])
    grouped = groupByKey(mapped)
    result = []
    for key, values in grouped:
        res = reduce_func(key, values)
        if res is not None:
            result.append(res)
    return result

output = MapReduce(RECORDREADER, MAP, REDUCE)
print(output)

[(User(id=0, age=55, social_contacts=20, gender='male'), User(id=0, age=55, social_contacts=20, gender='male'))]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [78]:
data_R = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=33, gender='female', social_contacts=800)
]

data_S = [
    User(id=3, age=35, gender='female', social_contacts=800),
    User(id=4, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240)
]

def RECORDREADER_R():
    for user in data_R:
        yield (user.id, user)

def RECORDREADER_S():
    for user in data_S:
        yield (user.id, user)

def MAP_R(userId: int, user: User):
    yield (user, 'R')

def MAP_S(userId: int, user: User):
    yield (user, 'S')

def REDUCE(key: User, values: list):
    if values == ['R']:
        return (key, key)

def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]

def groupByKey(items):
    groups = {}
    for key, value in sorted(items, key=lambda x: x[0]):
        if key not in groups:
            groups[key] = []
        groups[key].append(value)
    return [(k, v) for k, v in groups.items()]

def MapReduce(reader_R, map_R, reader_S, map_S, reduce_func):
    records_R = reader_R()
    mapped_R = flatten([list(map_R(*record)) for record in records_R])

    records_S = reader_S()
    mapped_S = flatten([list(map_S(*record)) for record in records_S])

    all_mapped = mapped_R + mapped_S

    grouped = groupByKey(all_mapped)

    result = []
    for key, values in grouped:
        res = reduce_func(key, values)
        if res is not None:
            result.append(res)
    return result

output = MapReduce(RECORDREADER_R, MAP_R, RECORDREADER_S, MAP_S, REDUCE)
print(output)

[(User(id=0, age=55, social_contacts=20, gender='male'), User(id=0, age=55, social_contacts=20, gender='male')), (User(id=2, age=33, social_contacts=800, gender='female'), User(id=2, age=33, social_contacts=800, gender='female'))]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [79]:
data_1 = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=33, gender='female', social_contacts=800),
    User(id=3, age=35, gender='female', social_contacts=800)
]

data_2 = [
    User(id=4, age=55, gender='male', social_contacts=20),
    User(id=5, age=25, gender='female', social_contacts=240),
    User(id=6, age=33, gender='female', social_contacts=800),
    User(id=0, age=55, gender='male', social_contacts=20)
]

def RECORDREADER_1():
    for user in data_1:
        yield (None, user)

def RECORDREADER_2():
    for user in data_2:
        yield (None, user)

def MAP_1(_, user: User):
    key = (user.age, user.gender, user.social_contacts)
    yield (key, ('R', user))

def MAP_2(_, user: User):
    key = (user.age, user.gender, user.social_contacts)
    yield (key, ('S', user))

def REDUCE(key, values):
    r_values = [v[1] for v in values if v[0] == 'R']
    s_values = [v[1] for v in values if v[0] == 'S']

    result = []
    for r in r_values:
        for s in s_values:
            if r.age == s.age and r.gender == s.gender and r.social_contacts == s.social_contacts:
                result.append((r, s))
    return result

def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]

def groupByKey(items):
    groups = {}
    for key, value in sorted(items, key=lambda x: x[0]):
        if key not in groups:
            groups[key] = []
        groups[key].append(value)
    return [(k, v) for k, v in groups.items()]

def MapReduce(reader1, map1, reader2, map2, reduce_func):
    records1 = reader1()
    mapped1 = flatten([list(map1(*record)) for record in records1])

    records2 = reader2()
    mapped2 = flatten([list(map2(*record)) for record in records2])

    all_mapped = mapped1 + mapped2
    grouped = groupByKey(all_mapped)

    result = []
    for key, values in grouped:
        res = reduce_func(key, values)
        if res:
            result.extend(res)
    return result

output = MapReduce(RECORDREADER_1, MAP_1, RECORDREADER_2, MAP_2, REDUCE)
print(output)

[(User(id=1, age=25, social_contacts=240, gender='female'), User(id=5, age=25, social_contacts=240, gender='female')), (User(id=2, age=33, social_contacts=800, gender='female'), User(id=6, age=33, social_contacts=800, gender='female')), (User(id=0, age=55, social_contacts=20, gender='male'), User(id=4, age=55, social_contacts=20, gender='male')), (User(id=0, age=55, social_contacts=20, gender='male'), User(id=0, age=55, social_contacts=20, gender='male'))]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [80]:
data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=33, gender='female', social_contacts=800),
    User(id=3, age=35, gender='female', social_contacts=800)
]

def RECORDREADER():
    for user in data:
        yield (None, user)

def MAP(_, user: User):
    yield (user.gender, user.age)

def REDUCE_SUM(key, values):
    return (key, sum(values))

def REDUCE_MAX(key, values):
    return (key, max(values))

def REDUCE_AVG(key, values):
    return (key, sum(values) / len(values))

def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]

def groupByKey(items):
    groups = {}
    for key, value in sorted(items, key=lambda x: x[0]):
        if key not in groups:
            groups[key] = []
        groups[key].append(value)
    return [(k, v) for k, v in groups.items()]

def MapReduce(reader, map_func, reduce_func):
    records = reader()
    mapped = flatten([list(map_func(*record)) for record in records])
    grouped = groupByKey(mapped)
    return [reduce_func(key, values) for key, values in grouped]

print("Сумма возрастов по полу:")
print(MapReduce(RECORDREADER, MAP, REDUCE_SUM))

print("\nМаксимальный возраст по полу:")
print(MapReduce(RECORDREADER, MAP, REDUCE_MAX))

Сумма возрастов по полу:
[('female', 93), ('male', 55)]

Максимальный возраст по полу:
[('female', 35), ('male', 55)]


### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [81]:
import numpy as np

matrix = np.ones((3, 3))
vector = np.array([1, 0, 0, 0])

maps = 2
reducers = 1

def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for i in range(split.shape[0]):
            for j in range(split.shape[1]):
                yield ((i, j), (split[i, j], vector[j], split.shape[1]))

    split_size = int(np.ceil(len(matrix) / maps))

    for i in range(0, len(matrix), split_size):
        yield RECORDREADER(matrix[i: i + split_size])


def MAP(coordinates: [int, int], values: [int, int, int]):
    i, j = coordinates
    matrix_value, vector_value, cols = values
    yield ((i, cols), matrix_value * vector_value)


def REDUCE(keys: [int, int], products: Iterator[NamedTuple]):
    i, cols = keys
    products_list = list(products)
    result = float(sum(products_list) / (len(products_list) / cols))
    yield (i, result)


partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

9 key-value pairs were sent over a network.


[(0, [(0, 1.0), (1, 1.0)])]

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [82]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [84]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
    (j, k) = k1
    w = v1
    for i in range(I):
        yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  yield (i, k), sum(values)

Проверьте своё решение

In [85]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [86]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [91]:
I = 3
J = 4
K = 5 * 10

small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def RECORDREADER():
  for i in range(small_mat.shape[0]):
    for j in range(big_mat.shape[0]):
      for k in range(big_mat.shape[1]):
        yield (((i, j), small_mat[i, j]),((j, k), big_mat[j, k]))

def MAP(list_1, list_2):
  (k1, v1) = list_1
  (k2, v2) = list_2
  yield ((k1[0], k2[1]), v1 * v2)

def REDUCE(key, values):
  (i, k) = key
  yield (key, sum(values))

# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

Реализуйте перемножение матриц с использованием модельного кода MapReduce
Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [88]:
I = 3
J = 4
K = 5*10

maps = 2
reducers = 3

small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def INPUTFORMAT():
  def RECORDREADER(key, split):
    mat = []
    for i in range(split.shape[0]):
      for j in range(split.shape[1]):
        mat.append(((key, i, j), split[i, j]))
    return mat

  yield RECORDREADER('S', small_mat)
  yield RECORDREADER('B', big_mat)


def MAP(k, v):
  (mat, i, j) = k
  w = v
  if mat == 'S':
    yield (j, (mat, i, w))
  else:
    yield (i, (mat, j, w))

def REDUCE(_, values):
  small = [v for v in values if v[0] == 'S']
  big = [v for v in values if v[0] == 'B']
  for s in small:
    for b in big:
      yield ((s[1], b[1]), s[2] * b[2])

def INPUT_MUL():
  for j in joined:
    yield j[1]

def MAP_MUL(k1, v1):
  yield (k1, v1)

def REDUCE_MUL(key, values):
  res_val = 0
  for v in values:
    res_val += v
  yield (key, res_val)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
joined = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

mul_output = MapReduceDistributed(INPUT_MUL, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_output]

solution = []
for p in pre_result:
    for v in p[1]:
        solution.append(v)

212 key-value pairs were sent over a network.
600 key-value pairs were sent over a network.


In [90]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [98]:
I = 3
J = 4
K = 5*10

maps = 2
reducers = 3

small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def INPUTFORMAT():
    global maps
    def RECORDREADER(key, split):
        mat = []
        for i in range(split.shape[0]):
            for j in range(split.shape[1]):
                mat.append(((key, i, j), split[i, j]))
        return mat

    small = RECORDREADER('S', small_mat)
    split_size =  int(np.ceil(len(small)/maps))
    for i in range(0, len(small), split_size):
        yield small[i:i+split_size]

    big = RECORDREADER('B', big_mat)
    split_size =  int(np.ceil(len(big)/maps))
    for i in range(0, len(big), split_size):
      yield big[i:i+split_size]

def MAP(k, v):
    (mat, i, j) = k
    w = v
    if mat == 'S':
        yield (j, (mat, i, w))
    else:
        yield (i, (mat, j, w))

def REDUCE(key, values):
    small = [v for v in values if v[0] == 'S']
    big = [v for v in values if v[0] == 'B']
    for s in small:
        for b in big:
          yield ((s[1], b[1]), s[2] * b[2])

def INPUT_MUL():
    for j in joined:
        yield j[1]

def MAP_MUL(k1, v1):
    yield (k1, v1)

def REDUCE_MUL(key, values):
    res_val = 0
    for v in values:
      res_val += v
    yield (key, res_val)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
joined = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

mul_output = MapReduceDistributed(INPUT_MUL, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_output]

solution = []
for p in pre_result:
    for v in p[1]:
      solution.append(v)

212 key-value pairs were sent over a network.
600 key-value pairs were sent over a network.


In [99]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)

def asmatrix(reduce_output):
    reduce_output = list(reduce_output)
    I = max(i for ((i,k), vw) in reduce_output)+1
    K = max(k for ((i,k), vw) in reduce_output)+1
    mat = np.empty(shape=(I,K))
    for ((i,k), vw) in reduce_output:
        mat[i,k] = vw
    return mat

print(np.allclose(reference_solution, asmatrix(solution))) # should return true

True


Решение не будет работать, если RECORDREADER-ы генерируют случайное подмножество элементов, потому что asmatrix использует max(i) от существующих элементов, что даст неполную матрицу, пропущенные элементы приведут к неверному результату.